In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit import TimeSformer
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-18 16:42:48,905]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
validation_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_valid_set30s.dat","valid")
print(1)
train_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_train_set30s.dat","train")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)/120
        return fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[1],
    tagdata=train_set_data[3],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[1],
    tagdata=validation_set_data[3],
    transform=transform
)

print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint file is kept.
# Whenever a better validation MAE is found across all seeds,
# this same file is overwritten.
# from full_30s_divided_by_person_SCENE3.vivit_vst_timsformer336699.TimeSformer_with_r2plus1d_1.timesformer.models.vit import TimeSformer


best_model_path = "./save_timesformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mae = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(0):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        # Re-create DataLoaders for each seed so that the
        # shuffled training order is also controlled by seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Every seed starts from a newly initialized model.
        # ----------------------------------------------------
        model=TimeSformer()#num_frames=frame_nums,batchsz = batchsz
        model.to(device)

        criterion = torch.nn.L1Loss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mae = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()

                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            # Convert to Python float for comparisons / logging.
            current_val_mae = val_avg_loss.item()

            # ------------------------------------------------
            # Best result of the CURRENT seed
            # ------------------------------------------------
            if current_val_mae < seed_best_mae:
                seed_best_mae = current_val_mae
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds.
            #
            # Only this ONE file is kept. If a new global best
            # is found, the same file is overwritten.
            # ------------------------------------------------
            if current_val_mae < global_best_mae:
                global_best_mae = current_val_mae
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mae,
                }, best_model_path)

                logger.info(
                    "Global best updated | "
                    "seed: {} | epoch: {} | 1-MAE: {:.4f}".format(
                        seed,
                        i + 1,
                        1.0 - current_val_mae
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | 1-MAE: {:.4f} | "
                "1-val_avg_o_loss: {:.4f} | "
                "1-val_avg_c_loss: {:.4f} | "
                "1-val_avg_e_loss: {:.4f} | "
                "1-val_avg_a_loss: {:.4f} | "
                "1-val_avg_n_loss: {:.4f}".format(
                    seed,
                    i + 1,
                    1.0 - current_val_mae,
                    1.0 - val_avg_o_loss.item(),
                    1.0 - val_avg_c_loss.item(),
                    1.0 - val_avg_e_loss.item(),
                    1.0 - val_avg_a_loss.item(),
                    1.0 - val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping for the CURRENT seed.
            # Stop when validation MAE has not improved for
            # 20 consecutive epochs.
            # ------------------------------------------------
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best 1-MAE: {:.4f}".format(
                        seed,
                        i + 1,
                        seed_best_epoch,
                        1.0 - seed_best_mae
                    )
                )
                break

        # ----------------------------------------------------
        # Store only scalar results for this seed.
        # No per-seed model is retained in memory or on disk.
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mae": seed_best_mae,
            "best_1_mae": 1.0 - seed_best_mae
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MAE = {seed_best_mae:.6f} | "
            f"Best 1-MAE = {1.0 - seed_best_mae:.6f}"
        )

        # ----------------------------------------------------
        # Release memory before starting the next seed.
        # ----------------------------------------------------
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary
# ============================================================

all_1_mae = np.array(
    [result["best_1_mae"] for result in seed_results],
    dtype=np.float64
)

mean_1_mae = all_1_mae.mean()
std_1_mae = all_1_mae.std(ddof=1) if len(all_1_mae) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | "
        "MAE {:.6f} | 1-MAE {:.6f}".format(
            result["seed"],
            result["best_epoch"],
            result["best_mae"],
            result["best_1_mae"]
        )
    )

print("-" * 80)
print(f"Mean 1-MAE : {mean_1_mae:.6f}")
print(f"Std  1-MAE : {std_1_mae:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"1-MAE={1.0 - global_best_mae:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 42 | Train 1/120: 100%|██████████| 1031/1031 [04:04<00:00,  4.21it/s]
Seed 42 | Valid 1/120: 100%|██████████| 337/337 [00:28<00:00, 11.82it/s]
[2026-09-18 16:47:32,863]::4164933583::INFO::Global best updated | seed: 42 | epoch: 1 | 1-MAE: 0.8698
[2026-09-18 16:47:32,865]::4164933583::DEBUG::Seed: 42 | Epoch: 1 | 1-MAE: 0.8698 | 1-val_avg_o_loss: 0.7539 | 1-val_avg_c_loss: 0.8899 | 1-val_avg_e_loss: 0.8624 | 1-val_avg_a_loss: 0.9310 | 1-val_avg_n_loss: 0.9116
Seed 42 | Train 2/120: 100%|██████████| 1031/1031 [04:04<00:00,  4.21it/s]
Seed 42 | Valid 2/120: 100%|██████████| 337/337 [00:27<00:00, 12.04it/s]
[2026-09-18 16:52:09,473]::4164933583::INFO::Global best updated | seed: 42 | epoch: 2 | 1-MAE: 0.8861
[2026-09-18 16:52:09,475]::4164933583::DEBUG::Seed: 42 | Epoch: 2 | 1-MAE: 0.8861 | 1-val_avg_o_loss: 0.8709 | 1-val_avg_c_loss: 0.8556 | 1-val_avg_e_loss: 0.8731 | 1-val_avg_a_loss: 0.9307 | 1-val_avg_n_loss: 0.9002
Seed 42 | Train 3/120: 100%|██████████| 1031/1031 [04:05<00:00, 

Seed 42 finished | Best epoch = 60 | Best MAE = 0.087663 | Best 1-MAE = 0.912337

Start training | seed = 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 123 | Train 1/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.13it/s]
Seed 123 | Valid 1/120: 100%|██████████| 337/337 [00:30<00:00, 11.22it/s]
[2026-09-18 22:59:46,906]::4164933583::DEBUG::Seed: 123 | Epoch: 1 | 1-MAE: 0.8543 | 1-val_avg_o_loss: 0.9111 | 1-val_avg_c_loss: 0.7970 | 1-val_avg_e_loss: 0.8738 | 1-val_avg_a_loss: 0.9281 | 1-val_avg_n_loss: 0.7617
Seed 123 | Train 2/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.13it/s]
Seed 123 | Valid 2/120: 100%|██████████| 337/337 [00:30<00:00, 11.20it/s]
[2026-09-18 23:04:27,665]::4164933583::DEBUG::Seed: 123 | Epoch: 2 | 1-MAE: 0.8847 | 1-val_avg_o_loss: 0.8953 | 1-val_avg_c_loss: 0.8876 | 1-val_avg_e_loss: 0.8615 | 1-val_avg_a_loss: 0.8919 | 1-val_avg_n_loss: 0.8871
Seed 123 | Train 3/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.13it/s]
Seed 123 | Valid 3/120: 100%|██████████| 337/337 [00:30<00:00, 11.09it/s]
[2026-09-18 23:09:08,714]::4164933583::DEBUG::Seed: 123 | Epoch: 3 | 1-MAE: 0.9011 | 1-val_avg_o_loss: 0.9082 | 1-

Seed 123 finished | Best epoch = 8 | Best MAE = 0.088682 | Best 1-MAE = 0.911318

Start training | seed = 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


...

Seed 777 finished | Best epoch = 21 | Best MAE = 0.088545 | Best 1-MAE = 0.911455

Start training | seed = 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 1004 | Train 1/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.12it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 337/337 [00:30<00:00, 11.19it/s]
[2026-09-19 04:23:19,317]::4164933583::DEBUG::Seed: 1004 | Epoch: 1 | 1-MAE: 0.8842 | 1-val_avg_o_loss: 0.8377 | 1-val_avg_c_loss: 0.8662 | 1-val_avg_e_loss: 0.8974 | 1-val_avg_a_loss: 0.9195 | 1-val_avg_n_loss: 0.9003
Seed 1004 | Train 2/120: 100%|██████████| 1031/1031 [04:10<00:00,  4.11it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 337/337 [00:30<00:00, 11.11it/s]
[2026-09-19 04:28:01,618]::4164933583::DEBUG::Seed: 1004 | Epoch: 2 | 1-MAE: 0.9069 | 1-val_avg_o_loss: 0.9069 | 1-val_avg_c_loss: 0.9137 | 1-val_avg_e_loss: 0.8901 | 1-val_avg_a_loss: 0.9229 | 1-val_avg_n_loss: 0.9008
Seed 1004 | Train 3/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.13it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 337/337 [00:30<00:00, 11.05it/s]
[2026-09-19 04:32:42,911]::4164933583::DEBUG::Seed: 1004 | Epoch: 3 | 1-MAE: 0.9053 | 1-val_avg_o_loss: 0.

Seed 1004 finished | Best epoch = 53 | Best MAE = 0.086939 | Best 1-MAE = 0.913061

Start training | seed = 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 2024 | Train 1/120: 100%|██████████| 1031/1031 [04:08<00:00,  4.14it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 337/337 [00:29<00:00, 11.29it/s]
[2026-09-19 10:05:35,103]::4164933583::DEBUG::Seed: 2024 | Epoch: 1 | 1-MAE: 0.8693 | 1-val_avg_o_loss: 0.9020 | 1-val_avg_c_loss: 0.8635 | 1-val_avg_e_loss: 0.7841 | 1-val_avg_a_loss: 0.8960 | 1-val_avg_n_loss: 0.9007
Seed 2024 | Train 2/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.14it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 337/337 [00:29<00:00, 11.24it/s]
[2026-09-19 10:10:15,612]::4164933583::DEBUG::Seed: 2024 | Epoch: 2 | 1-MAE: 0.8998 | 1-val_avg_o_loss: 0.9031 | 1-val_avg_c_loss: 0.8932 | 1-val_avg_e_loss: 0.8865 | 1-val_avg_a_loss: 0.9220 | 1-val_avg_n_loss: 0.8941
Seed 2024 | Train 3/120: 100%|██████████| 1031/1031 [04:09<00:00,  4.14it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 337/337 [00:29<00:00, 11.28it/s]
[2026-09-19 10:14:56,025]::4164933583::DEBUG::Seed: 2024 | Epoch: 3 | 1-MAE: 0.8932 | 1-val_avg_o_loss: 0.

Seed 2024 finished | Best epoch = 15 | Best MAE = 0.088299 | Best 1-MAE = 0.911701

MULTI-SEED RESULTS
Seed   42 | Best Epoch  60 | MAE 0.087663 | 1-MAE 0.912337
Seed  123 | Best Epoch   8 | MAE 0.088682 | 1-MAE 0.911318
Seed  777 | Best Epoch  21 | MAE 0.088545 | 1-MAE 0.911455
Seed 1004 | Best Epoch  53 | MAE 0.086939 | 1-MAE 0.913061
Seed 2024 | Best Epoch  15 | MAE 0.088299 | 1-MAE 0.911701
--------------------------------------------------------------------------------
Mean 1-MAE : 0.911974
Std  1-MAE : 0.000723
Global best: seed=1004, epoch=53, 1-MAE=0.913061
Saved checkpoint: ./save_timesformer_folder/model_best_multiseed.pth
